In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import Docx2txtLoader


file_path = r"C:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\hardik_resume.pdf"
if file_path.endswith(".pdf"):
    loader = PyPDFLoader(file_path)
elif file_path.endswith(".txt"):
    loader = TextLoader(file_path)
elif file_path.endswith(".docx"):
    loader = Docx2txtLoader(file_path)
docs=loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
split_docs = text_splitter.split_documents(docs)

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(split_docs))]

vector_store.add_documents(documents=split_docs, ids=uuids)


retriever = vector_store.as_retriever(search_kwargs={"k": 6})


from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b")


class ResumeDetails(BaseModel):
    name: str = Field(description="Full name of the candidate. Use 'Not Found' if missing.")
    experience: str = Field(description="Work experience. Use 'Not Found' if missing.")
    skills: str = Field(description="Skills. Use 'Not Found' if missing.")
    education: str = Field(description="Education details. Use 'Not Found' if missing.")
    certifications: str = Field(description="Certifications. Use 'Not Found' if missing.")
    projects: str = Field(description="Projects. Use 'Not Found' if missing.")


output_parser= JsonOutputParser(pydantic_object=ResumeDetails)
template= """You are an digital assistant that extracts resume details from the given resume. Extract the following details from the resume:
1. Name
2. Work Experience
3. Skills
4. Education
5. Certifications
6. Projects

note- If any field is not explicitly present in the resume text, return "Not Found".
Do not leave fields empty.
Do not infer or guess.
The candidate's name is usually present at the top of the resume.
Look carefully for standalone names, email headers, or contact sections.

"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",template),
        ("human", "Extract the details from the following resume:\n\n {context}\n\n{format_instructions}"),
    ]
)
prompt=prompt.partial(format_instructions=output_parser.get_format_instructions())
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs}
    | prompt
    | llm
    | output_parser
)



In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
split_docs = text_splitter.split_documents(docs)

In [30]:
split_docs[0].page_content

'Hardik Kumar\nNew Delhi, India|9310644380|hardikkumar.work25@gmail.com |linkedin.com/in/hardik-kumar\nEducation\nGuru Gobind Singh Indraprastha University (GGSIPU)New Delhi, India\nBachelor of Technology in Computational Science Aug. 2024 – May 2028\nCambridge Foundation SchoolNew Delhi, India\nSenior Secondary School (CBSE) – 85% May 2024\nTechnical Skills\nLanguages: Python, C++,C\nAI/ML Frameworks: TensorFlow, PyTorch, scikit-learn, Hugging Face Transformers,\nAgentic AI & LLMs: Multi-agent systems, RAG pipelines, Agent orchestration, Prompt engineering, Fine-tuning,\nLangChain, LangGraph, LlamaIndex, Autogen, n8n, CrewAI, Google ADK, Agno, AWS,\nTools & Technologies: Git, Docker, REST APIs, FastAPI, Flask, Vector databases, AWS, Google Cloud'

In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [31]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [32]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(split_docs))]

vector_store.add_documents(documents=split_docs, ids=uuids)

['611a0b2c-1555-4139-8bf0-9692dd555a01',
 'df101e20-5c55-4f18-be3f-9c7cfa2b1c3b',
 '77eaff23-a09e-4f0b-9f7f-0f4c8c19e244',
 '2c8677a0-2a73-4682-8073-851d93b772cc',
 '67da8b8d-666d-44fe-9787-7895edb30414',
 'e7fb7150-f833-44dd-8cc0-9a5547bb9cc4']

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 6})


In [34]:
result=retriever.invoke("Experience with machine learning")

In [35]:
result

[Document(id='e7fb7150-f833-44dd-8cc0-9a5547bb9cc4', metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-17T09:59:11+00:00', 'author': '', 'keywords': '', 'moddate': '2026-01-17T09:59:11+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\hardi\\OneDrive\\Desktop\\ats_resume_scorer\\hardik_resume.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Achievements & Activities\nTechnical Content Creator 2024\n• Actively document AI/ML learning journey, system design patterns, and practical implementation insights through\ntechnical blog posts and project documentation\n• Focus on translating complex AI concepts into clear, actionable content for developer community\nOpen Source Contributions 2024\n• Contribute to AI/ML open-source projects, focusing on agent-based systems and practical AI tool

In [10]:
from langchain_groq import ChatGroq
import os

In [11]:
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b")

In [12]:
llm.invoke("Hello")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hello". We should respond politely. No special instructions. Use friendly greeting.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.082183194, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.003352578, 'prompt_tokens_details': None, 'queue_time': 0.054543452, 'total_time': 0.085535772}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c08e8-e1d3-74e1-b398-63b75a71bdb5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

In [39]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b")


class ResumeDetails(BaseModel):
    name: str = Field(description="Full name of the candidate. Use 'Not Found' if missing.")
    experience: str = Field(description="Work experience. Use 'Not Found' if missing.")
    skills: str = Field(description="Skills. Use 'Not Found' if missing.")
    education: str = Field(description="Education details. Use 'Not Found' if missing.")
    certifications: str = Field(description="Certifications. Use 'Not Found' if missing.")
    projects: str = Field(description="Projects. Use 'Not Found' if missing.")


output_parser= JsonOutputParser(pydantic_object=ResumeDetails)
template= """You are an digital assistant that extracts resume details from the given resume. Extract the following details from the resume:
1. Name
2. Work Experience
3. Skills
4. Education
5. Certifications
6. Projects

note- If any field is not explicitly present in the resume text, return "Not Found".
Do not leave fields empty.
Do not infer or guess.
The candidate's name is usually present at the top of the resume.
Look carefully for standalone names, email headers, or contact sections.

"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",template),
        ("human", "Extract the details from the following resume:\n\n {context}\n\n{format_instructions}"),
    ]
)
prompt=prompt.partial(format_instructions=output_parser.get_format_instructions())
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs}
    | prompt
    | llm
    | output_parser
)


In [37]:
response = chain.invoke("resume of machine learning engineer")


In [38]:
response

{'name': 'Hardik Kumar',
 'experience': 'Not Found',
 'skills': 'Languages: Python, C++, C; AI/ML Frameworks: TensorFlow, PyTorch, scikit-learn, Hugging Face Transformers; Agentic AI & LLMs: Multi-agent systems, RAG pipelines, Agent orchestration, Prompt engineering, Fine-tuning, LangChain, LangGraph, LlamaIndex, Autogen, n8n, CrewAI, Google ADK, Agno, AWS; Tools & Technologies: Git, Docker, REST APIs, FastAPI, Flask, Vector databases, AWS, Google Cloud',
 'education': 'Guru Gobind Singh Indraprastha University (GGSIPU), New Delhi, India – Bachelor of Technology in Computational Science (Aug. 2024 – May 2028); Cambridge Foundation School, New Delhi, India – Senior Secondary School (CBSE) – 85% (May 2024)',
 'certifications': 'Not Found',
 'projects': 'ATS Resume Scorer & Job Matching System (2026): End-to-end ATS resume scoring system with multi-format support, deterministic scoring, job description analysis, improvement recommendations, and interactive Streamlit frontend. Multi-Agent 

In [2]:
from autogen_agentchat.agents import AssistantAgent
from models.openrouter import getOpenRouterModel

from autogen_agentchat.agents import AssistantAgent
from pydantic import BaseModel, Field
from typing import List, Optional

# Define proper data model
class JDAnalysis(BaseModel):
    job_title: str = Field(description="Job title")
    company: Optional[str] = Field(default=None, description="Company name if mentioned")
    location: Optional[str] = Field(default=None, description="Job location")
    
    # Skills breakdown
    must_have_skills: List[str] = Field(default_factory=list, description="Explicitly required skills")
    nice_to_have_skills: List[str] = Field(default_factory=list, description="Preferred/optional skills")
    
    # Experience
    experience_level: str = Field(description="Entry/Mid/Senior level")
    min_experience_years: Optional[int] = Field(default=None, description="Minimum years required")
    max_experience_years: Optional[int] = Field(default=None, description="Maximum years if range given")
    
    # Education
    required_education: str = Field(default="Not Specified", description="Minimum education requirement")
    preferred_education: Optional[str] = Field(default=None, description="Preferred education")
    
    # Certifications
    required_certifications: List[str] = Field(default_factory=list)
    preferred_certifications: List[str] = Field(default_factory=list)
    
    # Responsibilities
    key_responsibilities: List[str] = Field(default_factory=list, description="Main job duties")
    
    # Keywords categorized
    technical_keywords: List[str] = Field(default_factory=list, description="Technical terms, tools, frameworks")
    soft_skills_keywords: List[str] = Field(default_factory=list, description="Communication, leadership, etc.")
    domain_keywords: List[str] = Field(default_factory=list, description="Industry-specific terms")
    
    # Additional
    remote_policy: Optional[str] = Field(default=None, description="Remote/Hybrid/Onsite")
    salary_range: Optional[str] = Field(default=None, description="If mentioned")

SYSTEM_MESSAGE = """
You are a JD analyst agent that extracts structured information from job descriptions.

**Critical Instructions:**
1. Parse the job description carefully and thoroughly
2. Extract ONLY explicitly stated information
3. Do NOT infer, assume, or guess
4. Distinguish between "required" (must-have) and "preferred" (nice-to-have)
5. Return ONLY valid JSON matching the exact schema provided
6. Do not include markdown formatting, explanations, or preamble

**Schema Field Guidelines:**

- `must_have_skills`: Skills marked as "required", "must have", "mandatory"
- `nice_to_have_skills`: Skills marked as "preferred", "nice to have", "bonus", "plus"
- `experience_level`: Use exact terms: "Entry", "Mid", "Senior", "Lead", "Principal"
- `min_experience_years`: Minimum years stated (0 for fresher/entry-level)
- `max_experience_years`: If range like "0-2 years", use 2 as max
- `technical_keywords`: Programming languages, frameworks, tools, technologies
- `soft_skills_keywords`: Teamwork, communication, problem-solving, leadership
- `domain_keywords`: Industry terms like "ML", "NLP", "RAG", "ATS systems"

**Example Parsing:**
Input: "Required: Python, ML experience. Preferred: Docker knowledge. 0-2 years experience."
Output:
{
  "must_have_skills": ["Python", "Machine Learning"],
  "nice_to_have_skills": ["Docker"],
  "min_experience_years": 0,
  "max_experience_years": 2,
  "experience_level": "Entry"
}

Return valid JSON only. No explanations.
"""



model=getOpenRouterModel()
data_analyst_agent = AssistantAgent(
    name="JDAnalystAgent",
    model_client=model,
    description="An agent that analyzes job descriptions and provides insights.",
    system_message=SYSTEM_MESSAGE)

c:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\venv\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [16]:
task="""Job Title: AI / Machine Learning Engineer (Entry-Level)

Location: Remote / Bangalore
Experience Required: 0–2 Years

Role Overview

We are hiring an entry-level AI / Machine Learning Engineer to work on intelligent systems involving data processing, model training, and AI-powered applications. The role focuses on practical implementation of ML and NLP solutions rather than pure research.

Key Responsibilities

Build and deploy machine learning models using Python

Work on NLP tasks such as text classification and information extraction

Assist in developing AI pipelines for document processing and scoring systems

Integrate ML models with backend services and APIs

Perform data cleaning, preprocessing, and feature engineering

Required Skills

Proficiency in Python

Understanding of machine learning fundamentals

Experience with scikit-learn, TensorFlow, or PyTorch

Knowledge of NLP concepts and transformer-based models

Familiarity with vector databases or embeddings (FAISS, Pinecone, etc.)

Preferred Qualifications

Experience working with LangChain, RAG pipelines, or LLMs

Exposure to resume parsing or ATS systems

Knowledge of SQL or NoSQL databases

Familiarity with Docker or basic cloud deployment

Education

B.Tech / B.E. in Computer Science, AI, Data Science, or related fields

Additional Notes

This role is suitable for fresh graduates or final-year students with strong project experience in AI and machine learning."""

In [19]:
response=await data_analyst_agent.run(task=task)

In [24]:
response.messages[1].content

'{\n  "must_have_skills": [\n    "Python",\n    "Machine Learning fundamentals",\n    "scikit-learn",\n    "TensorFlow",\n    "PyTorch",\n    "NLP concepts and transformer-based models",\n    "vector databases or embeddings"\n  ],\n  "nice_to_have_skills": [\n    "LangChain, RAG pipelines, or LLMs",\n    "resume parsing or ATS systems",\n    "SQL or NoSQL databases",\n    "Docker or basic cloud deployment"\n  ],\n  "experience_level": "Entry",\n  "min_experience_years": 0,\n  "max_experience_years": 2,\n  "technical_keywords": [\n    "Python",\n    "scikit-learn",\n    "TensorFlow",\n    "PyTorch",\n    "NLP",\n    "Transformer models",\n    "Vector databases",\n    "FAISS",\n    "Pinecone",\n    "SQL",\n    "NoSQL",\n    "Docker",\n    "cloud deployment"\n  ],\n  "soft_skills_keywords": [],\n  "domain_keywords": [\n    "ML",\n    "NLP",\n    "RAG",\n    "ATS systems"\n  ]\n}'

In [71]:
result.messages[1]

TextMessage(id='4ea04c59-22ae-4584-8485-c452b33558f9', source='JDAnalystAgent', models_usage=RequestUsage(prompt_tokens=489, completion_tokens=1368), metadata={}, created_at=datetime.datetime(2026, 1, 29, 9, 44, 39, 567054, tzinfo=datetime.timezone.utc), content='{\n  "required_skills": [\n    "Proficiency in Python",\n    "Understanding of machine learning fundamentals",\n    "Experience with scikit-learn, TensorFlow, or PyTorch",\n    "Knowledge of NLP concepts and transformer-based models",\n    "Familiarity with vector databases or embeddings (FAISS, Pinecone, etc.)"\n  ],\n  "preferred_skills": [\n    "Experience working with LangChain, RAG pipelines, or LLMs",\n    "Exposure to resume parsing or ATS systems",\n    "Knowledge of SQL or NoSQL databases",\n    "Familiarity with Docker or basic cloud deployment"\n  ],\n  "experience_level": "Entry-Level",\n  "experience_years": 2,\n  "education": "B.Tech / B.E. in Computer Science, AI, Data Science, or related fields",\n  "keywords":

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from uuid import uuid4
from langchain_groq import ChatGroq
import os
from models.groq_openai import GetGroqModelClient
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from autogen_core.tools import FunctionTool
from autogen_agentchat.agents import AssistantAgent
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()

class ResumeDetails(BaseModel):
    name: str = Field(description="Full name of the candidate. Use 'Not Found' if missing.")
    experience: str = Field(description="Work experience. Use 'Not Found' if missing.")
    skills: str = Field(description="Skills. Use 'Not Found' if missing.")
    education: str = Field(description="Education details. Use 'Not Found' if missing.")
    certifications: str = Field(description="Certifications. Use 'Not Found' if missing.")
    projects: str = Field(description="Projects. Use 'Not Found' if missing.")

def load_resume(file_path: str):
    if file_path.endswith(".pdf"):
        return PyPDFLoader(file_path)
    elif file_path.endswith(".docx"):
        return Docx2txtLoader(file_path)
    elif file_path.endswith(".txt"):
        return TextLoader(file_path)
    else:
        raise ValueError("Unsupported resume format")

def process_resume(resume_path: str) -> dict:
    """Extract structured data from resume - NO RAG NEEDED"""
    
    loader = load_resume(resume_path)
    docs = loader.load()
    
    full_text = "\n\n".join([doc.page_content for doc in docs])
    
    llm = GetGroqModelClient()
    output_parser = JsonOutputParser(pydantic_object=ResumeDetails)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract resume details..."),
        ("human", "Resume text:\n\n{resume_text}\n\n{format_instructions}")
    ])
    
    chain = prompt | llm | output_parser
    
    return chain.invoke({
        "resume_text": full_text,
        "format_instructions": output_parser.get_format_instructions()
    })

resume_extractor_tool = FunctionTool(
    process_resume,
    description="Extracts structured information from a resume file and returns JSON with candidate details"
)
SYSTEM_MESSAGE = """You are a Resume Processor Agent.

Your job is to help extract resume information using your resume extraction tool.

When asked to process a resume:
1. Use the extract_resume_with_langchain tool with the file path
2. The tool will return structured JSON data
3. Analyze the returned data for completeness
4. Report the extracted information to the user

If the tool returns an error, explain what went wrong and suggest fixes.
"""
resume_agent=AssistantAgent(
    name="ResumeProcessor",
    model_client=model,
    tools=[resume_extractor_tool],
    description="Processes resumes and extracts structured candidate information",
    system_message=SYSTEM_MESSAGE,
    reflect_on_tool_use=True) 



In [40]:
file_path = r"C:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\hardik_resume.pdf"
response= await resume_agent.run(task=file_path)

In [43]:
response.messages[2].content

[FunctionExecutionResult(content="{'name': 'Hardik Kumar', 'experience': '2025: Multi-Agent AI System for Automated Workflows – Designed and deployed an end‑to‑end multi‑agent system coordinating specialized AI agents for task decomposition, execution, and validation across complex workflows.\\\\n2025: Pharmaceutical Agent – Built a production‑ready Retrieval‑Augmented Generation system with agentic decision‑making for intelligent document retrieval and response generation.\\\\n2025: Applied Machine Learning for Predictive Analytics – Developed supervised learning models (Random Forest, XGBoost, Neural Networks) achieving 88% accuracy on classification tasks with imbalanced datasets, including feature engineering, hyperparameter tuning, and automated data pipelines.\\\\n2026: ATS Resume Scorer & Job Matching System – Built an end‑to‑end ATS resume scoring system supporting multiple file formats, deterministic scoring, job description analysis, improvement recommendations, and an intera

In [4]:
from autogen_agentchat.agents import AssistantAgent
from models.openrouter import getOpenRouterModel

SYSTEM_MESSAGE = """
You are a JD analyst agent that extracts structured information from job descriptions.

Your task:
- Parse the given job description.
- Extract only information explicitly stated.
- Do not infer or guess.

Return ONLY a valid JSON object.
Do not include explanations, markdown, or extra text.

JSON schema:
{
  "required_skills": [string],
  "preferred_skills": [string],
  "experience_level": string,
  "experience_years": number or null,
  "education": string or "Not Specified",
  "keywords": [string]
}

Rules:
- If years are given as a range (e.g. 0–2), use the upper bound.
- If experience is described as "fresher" or "entry-level", set experience_years to 0.
- If a field is not present, use "Not Specified" or null as appropriate.
"""

def JDAnalystAgent(model):
    jd_analyst_agent = AssistantAgent(
        name="JDAnalystAgent",
        model_client=model,
        description="An agent that analyzes job descriptions and extracts information as structured JSON.",
        system_message=SYSTEM_MESSAGE)
    return jd_analyst_agent

task="""Job Title: AI / Machine Learning Engineer (Entry-Level)

Location: Remote / Bangalore
Experience Required: 0–2 Years

Role Overview

We are hiring an entry-level AI / Machine Learning Engineer to work on intelligent systems involving data processing, model training, and AI-powered applications. The role focuses on practical implementation of ML and NLP solutions rather than pure research.

Key Responsibilities

Build and deploy machine learning models using Python

Work on NLP tasks such as text classification and information extraction

Assist in developing AI pipelines for document processing and scoring systems

Integrate ML models with backend services and APIs

Perform data cleaning, preprocessing, and feature engineering

Required Skills

Proficiency in Python

Understanding of machine learning fundamentals

Experience with scikit-learn, TensorFlow, or PyTorch

Knowledge of NLP concepts and transformer-based models

Familiarity with vector databases or embeddings (FAISS, Pinecone, etc.)

Preferred Qualifications

Experience working with LangChain, RAG pipelines, or LLMs

Exposure to resume parsing or ATS systems

Knowledge of SQL or NoSQL databases

Familiarity with Docker or basic cloud deployment

Education

B.Tech / B.E. in Computer Science, AI, Data Science, or related fields

Additional Notes

This role is suitable for fresh graduates or final-year students with strong project experience in AI and machine learning."""



In [32]:
result.messages[1].content

'{\n  "required_skills": [\n    "Proficiency in Python",\n    "Understanding of machine learning fundamentals",\n    "Experience with scikit-learn, TensorFlow, or PyTorch",\n    "Knowledge of NLP concepts and transformer-based models",\n    "Familiarity with vector databases or embeddings (FAISS, Pinecone, etc.)"\n  ],\n  "preferred_skills": [\n    "Experience working with LangChain, RAG pipelines, or LLMs",\n    "Exposure to resume parsing or ATS systems",\n    "Knowledge of SQL or NoSQL databases",\n    "Familiarity with Docker or basic cloud deployment"\n  ],\n  "experience_level": "Entry-Level",\n  "experience_years": 2,\n  "education": "B.Tech / B.E. in Computer Science, AI, Data Science, or related fields",\n  "keywords": [\n    "AI",\n    "Machine Learning",\n    "NLP",\n    "Transformer",\n    "Vector Databases",\n    "FAISS",\n    "Pinecone",\n    "LangChain",\n    "RAG",\n    "LLMs",\n    "Resume Parsing",\n    "ATS",\n    "SQL",\n    "NoSQL",\n    "Docker",\n    "Cloud Deplo

In [ ]:
import asyncio
from config.docker import getDocker, startDocker, stopDocker
from models.openrouter import getOpenRouterModel
from teams.data_team import getTeamA
from agents.resume_processor import getResumeAgent
from teams.insights import getTeamInsights

async def main():
    model=getOpenRouterModel()
    docker=getDocker()
    resume_path=r"C:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\hardik_resume.pdf"
    resume_loader=getResumeAgent(model)
    data_team=getTeamA(model)
    insights_team=getTeamInsights(model)

    job_description="""Job Title: AI / Machine Learning Engineer (Entry-Level)

Location: Remote / Bangalore
Experience Required: 0–2 Years

Role Overview

We are hiring an entry-level AI / Machine Learning Engineer to work on intelligent systems involving data processing, model training, and AI-powered applications. The role focuses on practical implementation of ML and NLP solutions rather than pure research.

Key Responsibilities

Build and deploy machine learning models using Python

Work on NLP tasks such as text classification and information extraction

Assist in developing AI pipelines for document processing and scoring systems

Integrate ML models with backend services and APIs

Perform data cleaning, preprocessing, and feature engineering

Required Skills

Proficiency in Python

Understanding of machine learning fundamentals

Experience with scikit-learn, TensorFlow, or PyTorch

Knowledge of NLP concepts and transformer-based models

Familiarity with vector databases or embeddings (FAISS, Pinecone, etc.)

Preferred Qualifications

Experience working with LangChain, RAG pipelines, or LLMs

Exposure to resume parsing or ATS systems

Knowledge of SQL or NoSQL databases

Familiarity with Docker or basic cloud deployment

Education

B.Tech / B.E. in Computer Science, AI, Data Science, or related fields

Additional Notes

This role is suitable for fresh graduates or final-year students with strong project experience in AI and machine learning."""

    try:
        task = f"""
Process this resume: {resume_path}
Analyze this job description: {job_description}
Then score the match.
"""
        response=await data_team.run(task=task)
        print(response)
    except Exception as e:
        print(e)
    finally:
        exit()


RuntimeError: asyncio.run() cannot be called from a running event loop

In [15]:
data_team=getResumeAgent(model)
resume_path=r"C:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\hardik_resume.pdf"
task = f"""
Process this resume: {resume_path}"""

In [16]:
result=await data_team.run(task=task)

In [2]:
from agents.resume_processor import process_resume
resume_path=r"C:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\hardik_resume.pdf"
resume_details= process_resume(resume_path)

In [3]:
resume_details

{'name': 'Hardik Kumar',
 'experience': 'Worked on multiple AI/ML projects including ATS Resume Scorer & Job Matching System (2026), Multi-Agent AI System for Automated Workflows (2025), Pharmaceutical Agent (2025), and Applied Machine Learning for Predictive Analytics (2025). Responsibilities included system design, implementation, orchestration of multi-agent workflows, model development, and deployment of production-ready solutions.',
 'skills': 'Python, C++, C, TensorFlow, PyTorch, scikit-learn, Hugging Face Transformers, Multi-agent systems, RAG pipelines, Agent orchestration, Prompt engineering, Fine-tuning, LangChain, LangGraph, LlamaIndex, Autogen, n8n, CrewAI, Google ADK, Agno, AWS, Git, Docker, REST APIs, FastAPI, Flask, Vector databases, Google Cloud, Data Structures, Algorithms, Object-Oriented Programming, System Design, Backend Development',
 'education': 'Guru Gobind Singh Indraprastha University (GGSIPU), New Delhi, India – Bachelor of Technology in Computational Scienc